# Day 065 — Exercise 4: Write Deploy Files

A deployable app needs four files that most developers write manually and get slightly wrong each time. Generate them programmatically from a single function call — consistent, repeatable, and easy to add to a project generator.

| File | Purpose |
|------|---------|
| `requirements.txt` | Package list for `pip install -r` |
| `Procfile` | Start command for Render/Railway/Heroku |
| `render.yaml` | Render service definition (IaC) |
| `.env.example` | Documents every required environment variable |

In [ ]:
from pathlib import Path

_RENDER_YAML_TMPL = 'services:\n  - type: web\n    name: {app_name}\n    runtime: python\n    buildCommand: pip install -r requirements.txt\n    startCommand: uvicorn {module}:app --host 0.0.0.0 --port $PORT\n    envVars:\n      - key: MODEL\n        value: llama3.2\n      - key: STRIPE_SECRET_KEY\n        sync: false\n      - key: STRIPE_WEBHOOK_SECRET\n        sync: false\n'

_ENV_EXAMPLE = '# Copy to .env and fill in values. Never commit .env to git.\nMODEL=llama3.2\nPORT=8000\nSTRIPE_SECRET_KEY=sk_test_...\nSTRIPE_WEBHOOK_SECRET=whsec_...\n'


## Task

Implement `write_deploy_files(directory, app_name, module, packages) -> list[str]`:

- Default packages: `['fastapi', 'uvicorn[standard]', 'ollama', 'httpx']`
- `requirements.txt`: `'\n'.join(packages) + '\n'`
- `Procfile`: `f'web: uvicorn {module}:app --host 0.0.0.0 --port $PORT\n'`
- `render.yaml`: fill `_RENDER_YAML_TMPL` with `app_name` and `module`
- `.env.example`: write `_ENV_EXAMPLE` verbatim
- Return list of filenames created

## Your Implementation

In [ ]:
def write_deploy_files(directory: str, app_name: str = "my-app",
                       module: str = "app",
                       packages: list[str] | None = None) -> list[str]:
    """Write deployment configuration files to directory.

    Files created:
        requirements.txt  — one package per line (packages arg, or defaults)
        Procfile          — web: uvicorn {module}:app --host 0.0.0.0 --port $PORT
        render.yaml       — Render service definition (use _RENDER_YAML_TMPL)
        .env.example      — environment variable template (use _ENV_EXAMPLE)

    packages default: ['fastapi', 'uvicorn[standard]', 'ollama', 'httpx']
    Returns list of filenames written.
    """
    # TODO: build each file content, write to directory, return list of names
    raise NotImplementedError


In [ ]:
def write_deploy_files(directory: str, app_name: str = "my-app",
                       module: str = "app",
                       packages: list[str] | None = None) -> list[str]:
    base = Path(directory)
    base.mkdir(parents=True, exist_ok=True)
    pkgs = packages or ["fastapi", "uvicorn[standard]", "ollama", "httpx"]

    files = {
        "requirements.txt": "\n".join(pkgs) + "\n",
        "Procfile":         f"web: uvicorn {module}:app --host 0.0.0.0 --port $PORT\n",
        "render.yaml":      _RENDER_YAML_TMPL.format(app_name=app_name, module=module),
        ".env.example":     _ENV_EXAMPLE,
    }
    for fname, content in files.items():
        (base / fname).write_text(content, encoding="utf-8")
    return list(files.keys())


## Automated checks

In [ ]:
score, total = 0, 6
try:
    import tempfile
    from pathlib import Path

    with tempfile.TemporaryDirectory() as tmpdir:
        created = write_deploy_files(tmpdir, app_name="test-app", module="app")

        # returns a list of at least 4 filenames
        assert isinstance(created, list) and len(created) >= 4
        score += 1; print("\u2705 returns list of at least 4 filenames")

        # all files exist
        for f in created:
            assert (Path(tmpdir) / f).exists(), f"{f} missing"
        score += 1; print("\u2705 all listed files exist on disk")

        # requirements.txt has fastapi
        req = (Path(tmpdir) / "requirements.txt").read_text()
        assert "fastapi" in req.lower()
        score += 1; print("\u2705 requirements.txt contains fastapi")

        # Procfile references $PORT and uvicorn
        proc = (Path(tmpdir) / "Procfile").read_text()
        assert "uvicorn" in proc and "$PORT" in proc
        score += 1; print("\u2705 Procfile has uvicorn and $PORT")

        # render.yaml references the app_name
        render = (Path(tmpdir) / "render.yaml").read_text()
        assert "test-app" in render
        score += 1; print("\u2705 render.yaml contains app_name")

        # .env.example has MODEL=
        env_ex = (Path(tmpdir) / ".env.example").read_text()
        assert "MODEL=" in env_ex
        score += 1; print("\u2705 .env.example contains MODEL=")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def write_deploy_files(directory: str, app_name: str = "my-app",
                       module: str = "app",
                       packages: list[str] | None = None) -> list[str]:
    base = Path(directory)
    base.mkdir(parents=True, exist_ok=True)
    pkgs = packages or ["fastapi", "uvicorn[standard]", "ollama", "httpx"]

    files = {
        "requirements.txt": "\n".join(pkgs) + "\n",
        "Procfile":         f"web: uvicorn {module}:app --host 0.0.0.0 --port $PORT\n",
        "render.yaml":      _RENDER_YAML_TMPL.format(app_name=app_name, module=module),
        ".env.example":     _ENV_EXAMPLE,
    }
    for fname, content in files.items():
        (base / fname).write_text(content, encoding="utf-8")
    return list(files.keys())
```

**Why `.env.example` not `.env`?** `.env` is in `.gitignore` because it contains real secrets. `.env.example` is committed — it documents what variables are required without exposing values. New team members copy it to `.env` and fill in their own credentials.

</details>